# Unit 10 — ETF & Mutual Fund Analysis 🟡 IMPORTANT
**Phase 2 | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Most quant tutorials only work with price series. This unit goes deeper:
- **Holdings analysis** — what does an ETF actually own?
- **Expense ratios** — friction costs that compound over time
- **Overlap analysis** — are "diversified" ETFs actually holding the same stocks?
- **Sector exposures** — growth vs value vs blend

This is real portfolio management work. It also produces a clean portfolio project that stands out because most candidates only do price analysis.

> 💡 `yfinance`'s `.funds_data` attribute is a hidden gem — most tutorials never mention it.


---
## 1️⃣ The `.funds_data` API

`yf.Ticker(ticker).funds_data` returns a `FundsData` object with attributes:

| Attribute | Contains |
|-----------|----------|
| `.description` | Fund overview text |
| `.fund_overview` | Expense ratio, inception date, AUM |
| `.fund_operations` | Annual return data |
| `.top_holdings` | Top 10 holdings with weights |
| `.equity_holdings` | P/E, P/B, earnings growth breakdown |
| `.bond_holdings` | Duration, credit quality (for bond funds) |
| `.sector_weightings` | Sector % allocations |
| `.asset_classes` | Equity/bond/cash split |

> ⚠️ `funds_data` can be unreliable for some tickers — always wrap in try/except.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Our comparison universe
etfs = ['SPY', 'VOO', 'IVV', 'QQQ', 'VTI']

# Pull fund data for one ETF first
spy = yf.Ticker('SPY')

try:
    fd = spy.funds_data
    print("Available attributes:")
    print([attr for attr in dir(fd) if not attr.startswith('_')])
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Pull fund overview — expense ratios, AUM
def get_fund_overview(ticker):
    try:
        t = yf.Ticker(ticker)
        fd = t.funds_data
        overview = fd.fund_overview
        return overview
    except Exception as e:
        return {'error': str(e)}

print("SPY Fund Overview:")
spy_overview = get_fund_overview('SPY')
print(spy_overview)


In [ ]:
# Compare expense ratios across ETFs
def get_expense_ratio(ticker):
    try:
        fd = yf.Ticker(ticker).funds_data
        overview = fd.fund_overview
        # Expense ratio is typically in fund_overview dict
        if isinstance(overview, dict):
            return overview.get('annualReportExpenseRatio',
                   overview.get('annualHoldingsTurnover', None))
        return None
    except:
        return None

print("Expense Ratio Comparison:")
for etf in etfs:
    er = get_expense_ratio(etf)
    print(f"  {etf}: {er}")


---
## 2️⃣ Holdings Analysis

In [ ]:
# Get top holdings for each ETF
def get_holdings(ticker):
    try:
        fd = yf.Ticker(ticker).funds_data
        holdings = fd.top_holdings
        if holdings is not None and len(holdings) > 0:
            return holdings
        return pd.DataFrame()
    except Exception as e:
        return pd.DataFrame()

print("SPY Top Holdings:")
spy_holdings = get_holdings('SPY')
print(spy_holdings)


In [ ]:
# QQQ comparison (tech-heavy vs broad market)
print("QQQ Top Holdings:")
qqq_holdings = get_holdings('QQQ')
print(qqq_holdings)


In [ ]:
# Holdings overlap — how similar are SPY and VOO?
# Both track S&P 500 — should have near-identical holdings

def holdings_overlap(ticker1, ticker2):
    h1 = get_holdings(ticker1)
    h2 = get_holdings(ticker2)

    if h1.empty or h2.empty:
        return None

    # Find common column name for ticker symbols
    # yfinance may use different column names
    print(f"{ticker1} columns: {list(h1.columns)}")
    print(f"{ticker2} columns: {list(h2.columns)}")

    return h1, h2

h_spy, h_voo = holdings_overlap('SPY', 'VOO')


---
## 3️⃣ Sector Weightings

In [ ]:
# Sector exposure comparison — QQQ vs SPY
def get_sectors(ticker):
    try:
        fd = yf.Ticker(ticker).funds_data
        sectors = fd.sector_weightings
        if sectors is not None:
            return pd.Series(sectors, name=ticker)
        return pd.Series(name=ticker)
    except:
        return pd.Series(name=ticker)

sector_data = {}
for etf in ['SPY', 'QQQ', 'VTI']:
    s = get_sectors(etf)
    if not s.empty:
        sector_data[etf] = s

if sector_data:
    sector_df = pd.DataFrame(sector_data).fillna(0) * 100
    print("Sector Weightings (%):")
    print(sector_df.round(1).sort_values('SPY', ascending=False))


In [ ]:
# Price performance comparison — the bottom line
import matplotlib.pyplot as plt

prices = yf.download(etfs, start='2020-01-01', auto_adjust=True)['Close']
normalised = prices / prices.iloc[0] * 100  # Index to 100 at start

print("Cumulative Returns (indexed to 100):")
print(normalised.tail(3).round(2))

# Total returns
total_returns = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
print("\nTotal Returns since 2020-01-01:")
print(total_returns.round(1).sort_values(ascending=False))


---
## 4️⃣ Optional FRED Extension — Macro Context

Reinforce Unit 4 resampling skills by overlaying macro data on ETF prices.

**The pattern:** Monthly FRED data + Daily ETF data = frequency mismatch problem → solve with `.resample('D').ffill()`

**Key relationship to know:** Fed Funds Rate ↑ → Bond prices ↓ (inverse relationship)


In [ ]:
# Optional: FRED macro overlay (requires pandas-datareader)
# pip install pandas-datareader

try:
    import pandas_datareader as pdr

    # Pull Fed Funds Rate (monthly)
    fed_funds = pdr.get_data_fred('FEDFUNDS', start='2019-01-01')
    print("Fed Funds Rate (last 5):")
    print(fed_funds.tail(5))

    # Pull TLT (Long-term Treasury ETF — inversely correlated with rates)
    tlt = yf.download('TLT', start='2019-01-01', auto_adjust=True)['Close'].squeeze()

    # Resample Fed Funds to daily (forward-fill monthly values)
    # Why ffill? Because the rate is constant until changed — ffill is correct
    fed_daily = fed_funds.resample('D').ffill().reindex(tlt.index, method='ffill')

    # Correlation (should be negative — rates up = TLT down)
    corr = tlt.pct_change().corr(fed_daily['FEDFUNDS'].diff().dropna().reindex(tlt.index))
    print(f"\nCorrelation (TLT daily returns vs Fed Funds daily change): {corr:.3f}")
    print("→ Should be negative (rates rise → bond prices fall)")

except ImportError:
    print("pandas-datareader not installed. Run: pip install pandas-datareader")
except Exception as e:
    print(f"Error: {e}")


---
## ✅ Self-Check Questions

1. What does `auto_adjust=True` handle that `auto_adjust=False` doesn't?
   > *Both splits AND dividends — gives you total return series.*

2. What's the expected relationship between Fed Funds Rate and TLT (long Treasury ETF)?
   > *Inverse: rates rise → existing bond prices fall → TLT falls.*

3. Why would SPY and VOO have near-identical holdings but slightly different performance?
   > *Both track the S&P 500 but different expense ratios and different rebalancing methods create tiny tracking differences.*

4. Why forward-fill monthly FRED data to daily frequency?
   > *The Fed Funds Rate is constant between FOMC decisions — it doesn't change daily. Forward-filling correctly represents what the rate actually was on each day.*

---
## 🧪 Optional Tasks

- Compare expense ratios: SPY (0.0945%) vs VOO (0.03%) vs IVV (0.03%). Over 30 years on $100k, how much does the difference compound to?
- Plot QQQ vs SPY normalised to 100 from 2015. When did QQQ outperform most dramatically? Why? (Tech boom 2017-2021)
- If `funds_data` is working: compare sector weights of SPY vs QQQ. Compute the difference to see how QQQ is positioned.
- Implement the FRED extension: plot dual-axis chart of Fed Funds Rate vs TLT price (2019–present). Annotate rate hike cycles.
